# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

Each entity is referenced directly via its `@id` as per the Croissant schema.

In [ ]:
# List all record sets by their @id
if hasattr(metadata, 'record_sets'):
    print("Record Sets (@id):")
    for rset in metadata.record_sets:
        print(f"- {rset['@id']} (name: {rset.get('name', '')})")
else:
    print("No 'record_sets' attribute found in metadata, searching top-level...")
    # Sometimes Croissant metadata puts record sets in .recordSet
    if hasattr(metadata, 'recordSet') and metadata.recordSet:
        print("Record Sets (@id):")
        for rset in metadata.recordSet:
            if isinstance(rset, dict) and '@id' in rset:
                print(f"- {rset['@id']} (name: {rset.get('name', '')})")
    else:
        print("No record sets found in this dataset.")

# Try to find all fields (across all record sets)
def collect_fields(metadata):
    fields = []
    # Check possible attributes
    possible_entrypoints = ['record_sets', 'recordSet']
    for attr in possible_entrypoints:
        if hasattr(metadata, attr):
            rsets = getattr(metadata, attr)
            if isinstance(rsets, list):
                for rset in rsets:
                    # Sometimes fields are listed under different names
                    fields_cand = None
                    for key in ['fields', 'field', 'columns', 'column']:
                        if key in rset:
                            fields_cand = rset[key]
                            break
                    if fields_cand:
                        for field in fields_cand:
                            if isinstance(field, dict) and '@id' in field:
                                fields.append(field['@id'])
    return fields

fields = collect_fields(metadata)
print("\nSample of field/column @ids found:")
for _id in fields[:10]:
    print(f"- {_id}")

# If available, print field details for a record set
# Try to infer a record_set id to use later
record_set_ids = []
if hasattr(metadata, 'record_sets'):
    record_set_ids = [rset['@id'] for rset in metadata.record_sets]
elif hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_set_ids = [rset['@id'] for rset in metadata.recordSet if isinstance(rset, dict) and '@id' in rset]

if record_set_ids:
    print(f"\nSample record set @id to use: {record_set_ids[0]}")
else:
    print("No record set found to preview records.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field/column `@id`s from the overview section.

In [ ]:
# Prepare record set selection
record_sets = record_set_ids  # From the previous cell, list of record set @ids

dataframes = {}
for record_set in record_sets:
    print(f"Loading records for record set @id: {record_set}")
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df

if record_sets:
    main_record_set = record_sets[0]  # Use the first one for demonstration
    print(f"\nColumns found in DataFrame for record set {main_record_set}:")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping by attributes.

In [ ]:
# Attempt EDA on a numeric field (e.g., MW [Molecular Weight], Coverage_Percent, Peptide_Count)
import numpy as np

# Helper: select a likely numeric field (try common names in case-insensitive search)
if record_sets:
    df = dataframes[main_record_set]
    candidates = [col for col in df.columns if any(word in col.lower() for word in ['mw', 'weight', 'coverage', 'percent', 'intensity', 'count', 'abundance'])]
    if candidates:
        numeric_field = candidates[0]
    else:
        # fallback: first numeric-typed column
        numeric_field = df.select_dtypes(include=[np.number]).columns.tolist()[0] if not df.select_dtypes(include=[np.number]).empty else df.columns[0]
    print(f"Chosen numeric field for EDA: {numeric_field}")

    # Try to remove non-numeric items (convert if possible)
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].mean() # Use mean as example threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records in '{numeric_field}' > mean ({threshold:.2f}): {len(filtered_df)} records out of {len(df)}")

    # Normalize (z-score) the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a likely category/text field
    group_candidates = [col for col in df.columns if any(word in col.lower() for word in ['accession', 'protein', 'organism', 'sample', 'type', 'description']) and col != numeric_field]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"\nGrouping by field: {group_field}")
        # Show mean normalized value per group
        grouped_df = filtered_df.groupby(group_field)[f"{numeric_field}_normalized"].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No suitable text field found for grouping.")
else:
    print("No record sets/dataframes loaded for EDA.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and the mean per group if grouped.

In [ ]:
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[main_record_set][numeric_field].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped_df exists and has more than one value, plot barplot
    if 'grouped_df' in globals() and not grouped_df.empty and grouped_df.shape[0] < 50:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field, y=f"{numeric_field}_normalized")
        plt.title(f"Mean normalized {numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print("No data for visualization.")

## 6. Conclusion
This notebook demonstrated how to load a FAIR dataset defined by a Croissant schema using `mlcroissant`, inspect its structure, and perform basic exploration and visualization. 

**Key steps applied:**
* Loaded dataset using the Croissant schema URL referencing all entities by their `@id`.
* Explored available record sets and their fields.
* Extracted a sample record set to a pandas DataFrame.
* Filtered records, normalized data, and explored relationships.
* Visualized numeric field distributions and summary statistics.

This workflow can be adapted for other Croissant-structured datasets in the life sciences, omics, or beyond.